In [ ]:
#A look specifically at environmental burden with control for income and region

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt


df = pd.read_csv("/Users/obedfrimpong/Desktop/qss-45-final-project/EJI_2024_United_States.csv")
df = df.replace(-999, np.nan).copy()

needed = ["EPL_MINRTY", "EPL_POV200", "RPL_EBM", "RPL_SVM", "StateDesc", "STATEABBR"]
df = df.dropna(subset=needed).copy()

region_map = {
    "Northeast": [
        "Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island",
        "Vermont", "New Jersey", "New York", "Pennsylvania"
    ],
    "Midwest": [
        "Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin",
        "Iowa", "Kansas", "Minnesota", "Missouri", "Nebraska",
        "North Dakota", "South Dakota"
    ],
    "South": [
        "Delaware", "District of Columbia", "Florida", "Georgia", "Maryland",
        "North Carolina", "South Carolina", "Virginia", "West Virginia",
        "Alabama", "Kentucky", "Mississippi", "Tennessee",
        "Arkansas", "Louisiana", "Oklahoma", "Texas"
    ],
    "West": [
        "Arizona", "Colorado", "Idaho", "Montana", "Nevada", "New Mexico",
        "Utah", "Wyoming", "Alaska", "California", "Hawaii", "Oregon", "Washington"
    ]
}

state_to_region = {}
for region, states in region_map.items():
    for s in states:
        state_to_region[s] = region

df["Region"] = df["StateDesc"].map(state_to_region)
df = df.dropna(subset=["Region"]).copy()

# Main models

# Outcome options:
# RPLEBM = environmental burden rank
# RPLSVM = social vulnerability rank
# You can switch to other burden variables if you want
outcome = "RPL_EBM"

# Control for income and region
model = smf.ols(f"{outcome} ~ EPL_MINRTY + EPL_POV200 + C(Region)", data=df).fit(cov_type="HC3")
print(model.summary())

model_interaction = smf.ols(
    f"{outcome} ~ EPL_MINRTY * EPL_POV200 + C(Region)",
    data=df
).fit(cov_type="HC3")
print(model_interaction.summary())

# Summarize effect sizes
results = pd.DataFrame({
    "term": model.params.index,
    "coef": model.params.values,
    "std_err": model.bse.values,
    "p_value": model.pvalues.values
})
results.to_csv("eji_burden_regression_results.csv", index=False)

interaction_results = pd.DataFrame({
    "term": model_interaction.params.index,
    "coef": model_interaction.params.values,
    "std_err": model_interaction.bse.values,
    "p_value": model_interaction.pvalues.values
})
interaction_results.to_csv("eji_burden_regression_interaction_results.csv", index=False)

df["minority_bin"] = pd.qcut(df["EPL_MINRTY"], q=5, duplicates="drop")

df["minority_bin"] = df["minority_bin"].astype(str)

plot_df = df.groupby(["minority_bin", "Region"], observed=False)[outcome].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=plot_df, x="minority_bin", y=outcome, hue="Region", marker="o")
plt.xticks(rotation=45, ha="right")
plt.title(f"{outcome} by Minority Share and Region")
plt.xlabel("Minority share quintile")
plt.ylabel(f"Mean {outcome}")
plt.tight_layout()
plt.savefig("eji_burden_by_minority_region.png", dpi=150, bbox_inches="tight")
plt.close()

# Residualize outcome and minority share on controls
controls = smf.ols(f"{outcome} ~ EPL_POV200 + C(Region)", data=df).fit()
minority_ctrl = smf.ols("EPL_MINRTY ~ EPL_POV200 + C(Region)", data=df).fit()

df["outcome_resid"] = controls.resid
df["minority_resid"] = minority_ctrl.resid

partial_corr = df[["outcome_resid", "minority_resid"]].corr().iloc[0, 1]

print(f"Partial correlation: {partial_corr:.6f}")

                            OLS Regression Results                            
Dep. Variable:                RPL_EBM   R-squared:                       0.238
Model:                            OLS   Adj. R-squared:                  0.238
Method:                 Least Squares   F-statistic:                     6419.
Date:                Sun, 07 Jun 2026   Prob (F-statistic):               0.00
Time:                        01:01:12   Log-Likelihood:                -3354.3
No. Observations:               82740   AIC:                             6721.
Df Residuals:                   82734   BIC:                             6777.
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  0